# Analisis de precios
Fuente: Precios Claros - Base [SEPA](https://datos.produccion.gob.ar/dataset/sepa-precios)

In [1]:
# 1. Importar librerias

import zipfile
import io
import re
from pathlib import Path

import pandas as pd
from tqdm.notebook import tqdm

In [2]:
# 2. Deteccion archivos .zip

# Directorio de trabajo en Colab (donde se suben archivos por defecto)
BASE_DIR = Path("/content")

# Detectar todos los .zip que empiecen con "sepa_" en el directorio raíz
zips_encontrados = sorted(BASE_DIR.glob("sepa_*.zip"))

if not zips_encontrados:
    print("No se encontraron archivos sepa_*.zip en /content")
else:
    print(f"ZIPs detectados ({len(zips_encontrados)}):")
    for z in zips_encontrados:
        print(f"  {z.name}")

ZIPs detectados (1):
  sepa_martes.zip


In [3]:
# 3. Procesar extraccion

def extraer_csv_de_zip_interno(zip_bytes: bytes, nombre_zip: str) -> dict[str, pd.DataFrame]:
    """
    Abre un ZIP de comercio en memoria y devuelve un dict con los
    DataFrames de 'comercio', 'productos' y 'sucursales'.
    """
    dfs = {}
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zc:
        for nombre_csv in zc.namelist():
            nombre_lower = nombre_csv.lower()

            if "producto" in nombre_lower:
                clave = "productos"
            elif "sucursal" in nombre_lower:
                clave = "sucursales"
            elif "comercio" in nombre_lower:
                clave = "comercio"
            else:
                continue

            with zc.open(nombre_csv) as f:
                try:
                    df = pd.read_csv(
                        f,
                        sep="|",
                        dtype=str,
                        on_bad_lines="skip",
                        encoding="utf-8",
                        engine="python",
                    )
                    df.columns = df.columns.str.strip().str.lower()
                    dfs[clave] = df
                except Exception as e:
                    print(f"    [!] Error leyendo {nombre_csv} en {nombre_zip}: {e}")
    return dfs


def parsear_metadata_zip(nombre_zip: str) -> dict:
    """
    Extrae metadata del nombre del ZIP interno.
    Ej: sepa_1_comercio-sepa-4_2026-04-17_09-05-10
    """
    patron = r"sepa_(\d+)_comercio-sepa-(\d+)_(\d{4}-\d{2}-\d{2})_(\d{2}-\d{2}-\d{2})"
    m = re.search(patron, nombre_zip)
    if m:
        return {
            "sepa_version": int(m.group(1)),
            "id_comercio_zip": int(m.group(2)),
            "fecha_str": m.group(3),
            "hora_str": m.group(4).replace("-", ":"),
        }
    return {}

In [4]:
# 4. Procesar datos

def procesar_zips(lista_zips: list[Path]) -> dict[str, pd.DataFrame]:
    """
    Itera sobre los ZIPs detectados automáticamente y devuelve
    tres DataFrames unificados: comercio, productos y sucursales.
    """
    acum = {"comercio": [], "productos": [], "sucursales": []}

    for zip_path in lista_zips:
        # El "día" se deriva del nombre del archivo (sepa_lunes.zip → lunes)
        dia = zip_path.stem.replace("sepa_", "")
        print(f"\n Procesando: {zip_path.name}  (día: {dia})")

        try:
            with zipfile.ZipFile(zip_path) as zdia:
                # Detectar subcarpetas de fecha (ej: 2026-04-17/)
                carpetas_fecha = {
                    n.split("/")[0]
                    for n in zdia.namelist()
                    if re.match(r"\d{4}-\d{2}-\d{2}", n)
                }

                if not carpetas_fecha:
                    print(f"  [!] No se encontraron carpetas de fecha en {zip_path.name}")
                    continue

                for fecha_str in sorted(carpetas_fecha):
                    print(f"  Fecha: {fecha_str}")

                    zips_internos = [
                        n for n in zdia.namelist()
                        if n.startswith(fecha_str + "/") and n.endswith(".zip")
                    ]

                    if not zips_internos:
                        print(f"    [!] Sin ZIPs internos para {fecha_str}")
                        continue

                    for nombre_interno in tqdm(zips_internos, desc=f"  {fecha_str}", leave=False):
                        nombre_base = Path(nombre_interno).name
                        meta = parsear_metadata_zip(nombre_base)

                        with zdia.open(nombre_interno) as f:
                            zip_bytes = f.read()

                        dfs_comercio = extraer_csv_de_zip_interno(zip_bytes, nombre_base)

                        for clave, df in dfs_comercio.items():
                            df["dia"]            = dia
                            df["fecha"]          = fecha_str
                            df["sepa_version"]   = meta.get("sepa_version")
                            df["archivo_origen"] = nombre_base
                            acum[clave].append(df)

        except zipfile.BadZipFile:
            print(f"  [!] Archivo corrupto o no es un ZIP válido: {zip_path.name}")
        except Exception as e:
            print(f"  [!] Error inesperado en {zip_path.name}: {e}")

    # Concatenar acumulados
    resultado = {}
    for clave, lista in acum.items():
        if lista:
            resultado[clave] = pd.concat(lista, ignore_index=True)
            print(f"\n df_{clave}: {len(resultado[clave]):,} filas")
        else:
            resultado[clave] = pd.DataFrame()
            print(f"\n df_{clave}: vacío")

    return resultado


# ── EJECUTAR ──────────────────────────────────────────────────
datos = procesar_zips(zips_encontrados)

df_comercios  = datos["comercio"]
df_productos  = datos["productos"]
df_sucursales = datos["sucursales"]


 Procesando: sepa_martes.zip  (día: martes)
  Fecha: 2026-04-21


  2026-04-21:   0%|          | 0/7 [00:00<?, ?it/s]


 df_comercio: 14 filas

 df_productos: 456,400 filas

 df_sucursales: 196 filas


In [5]:
# 5. Tipado y limpieza

def limpiar_tipos(df_prod: pd.DataFrame, df_suc: pd.DataFrame):
    cols_precio = [
        "productos_precio_lista",
        "productos_precio_referencia",
        "productos_precio_unitario_promo1",
        "productos_precio_unitario_promo2",
        "productos_cantidad_presentacion",
        "productos_cantidad_referencia",
    ]
    for col in cols_precio:
        if col in df_prod.columns:
            df_prod[col] = pd.to_numeric(df_prod[col], errors="coerce")

    for col in ["id_comercio", "id_bandera", "id_sucursal"]:
        if col in df_prod.columns:
            df_prod[col] = pd.to_numeric(df_prod[col], errors="coerce").astype("Int64")

    df_prod["fecha"] = pd.to_datetime(df_prod["fecha"], errors="coerce")

    for col in ["sucursales_latitud", "sucursales_longitud"]:
        if col in df_suc.columns:
            df_suc[col] = pd.to_numeric(df_suc[col], errors="coerce")

    df_suc["fecha"] = pd.to_datetime(df_suc["fecha"], errors="coerce")

    return df_prod, df_suc


df_productos, df_sucursales = limpiar_tipos(df_productos, df_sucursales)
print("Tipos aplicados.")

Tipos aplicados.


In [6]:
# 6. Productos unicos

# Columnas que identifican un producto único
COLS_PRODUCTO = [
    "productos_ean",
    "productos_descripcion",
    "productos_marca",
    "productos_unidad_medida_presentacion",
    "productos_cantidad_presentacion",
]

# Filtrar solo las columnas que existen en el DataFrame
cols_presentes = [c for c in COLS_PRODUCTO if c in df_productos.columns]

# Deduplicar y ordenar
df_productos_unicos = (
    df_productos[cols_presentes]
    .drop_duplicates()
    .sort_values(["productos_marca", "productos_descripcion"])
    .reset_index(drop=True)
)

# Renombrar columnas para el Excel
df_productos_unicos.columns = (
    df_productos_unicos.columns
    .str.replace("productos_", "", regex=False)
    .str.replace("_", " ", regex=False)
    .str.title()
)

print(f"Productos únicos encontrados: {len(df_productos_unicos):,}")
display(df_productos_unicos.head(20))

# Exportar a Excel
output_path = BASE_DIR / "productos_unicos.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    df_productos_unicos.to_excel(writer, index=False, sheet_name="Productos")

    # Ajustar ancho de columnas automáticamente
    ws = writer.sheets["Productos"]
    for col in ws.columns:
        max_len = max(len(str(cell.value or "")) for cell in col)
        ws.column_dimensions[col[0].column_letter].width = min(max_len + 4, 60)

print(f"\nExportado: {output_path}")

Productos únicos encontrados: 31,431


,Ean,Descripcion,Marca,Unidad Medida Presentacion,Cantidad Presentacion
0,1,Contenedor Barbara 3 U 3 Lt,0,NaN,NaN
1,1,100 DUC.budín c/frutas 170 g (z),100 DUCADOS,NaN,NaN
2,1,100 DUC.budín manz/can. 170 g (z),100 DUCADOS,NaN,NaN
3,1,100 DUC.budín maracuya 170 g (z),100 DUCADOS,NaN,NaN
4,1,100 DUC.pan dulce trad. 700 g (z),100 DUCADOS,NaN,NaN
5,1,100 DUC.torta cioccolato 600 g (z),100 DUCADOS,NaN,NaN
6,1,100 DUCADOS budin c/chips choc. 300 g,100 DUCADOS,NaN,NaN
7,1,100 DUCADOS budin c/chips ddl 300 g,100 DUCADOS,NaN,NaN
8,1,100 DUCADOS budin c/dulce de leche 300 g,100 DUCADOS,NaN,NaN
9,1,100 DUCADOS budin c/frut.secos 300 g,100 DUCADOS,NaN,NaN



Exportado: /content/productos_unicos.xlsx


In [7]:
# Columnas de identificación del producto
COLS_ID = [
    "productos_ean",
    "productos_descripcion",
    "productos_marca",
    "productos_unidad_medida_presentacion",
    "productos_cantidad_presentacion",
]

# Filtrar solo las que existen
cols_id_presentes = [c for c in COLS_ID if c in df_productos.columns]

# Calcular precio promedio, mínimo y máximo por producto
df_precios = (
    df_productos
    .groupby(cols_id_presentes, dropna=False)
    .agg(
        precio_promedio=("productos_precio_lista", "mean"),
        precio_minimo=("productos_precio_lista", "min"),
        precio_maximo=("productos_precio_lista", "max"),
        cantidad_registros=("productos_precio_lista", "count"),
    )
    .reset_index()
    .sort_values(["productos_marca", "productos_descripcion"])
    .reset_index(drop=True)
)

# Redondear precios a 2 decimales
for col in ["precio_promedio", "precio_minimo", "precio_maximo"]:
    df_precios[col] = df_precios[col].round(2)

print(f"Productos únicos con precio: {len(df_precios):,}")
display(df_precios.head(20))

# Exportar a Excel
output_path = BASE_DIR / "productos_precios_promedio.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    df_precios.to_excel(writer, index=False, sheet_name="Precios")

    ws = writer.sheets["Precios"]

    # Ajustar ancho de columnas
    for col in ws.columns:
        max_len = max(len(str(cell.value or "")) for cell in col)
        ws.column_dimensions[col[0].column_letter].width = min(max_len + 4, 60)

    # Formato de número con 2 decimales para columnas de precio
    from openpyxl.styles import numbers
    col_indices = {cell.value: cell.column for cell in ws[1]}
    for nombre_col in ["precio_promedio", "precio_minimo", "precio_maximo"]:
        if nombre_col in col_indices:
            idx = col_indices[nombre_col]
            for row in ws.iter_rows(min_row=2, min_col=idx, max_col=idx):
                for cell in row:
                    cell.number_format = "#,##0.00"

print(f"\nExportado: {output_path}")

Productos únicos con precio: 31,431


,productos_ean,productos_descripcion,productos_marca,productos_unidad_medida_presentacion,productos_cantidad_presentacion,precio_promedio,precio_minimo,precio_maximo,cantidad_registros
0,1,Contenedor Barbara 3 U 3 Lt,0,NaN,NaN,NaN,NaN,NaN,0
1,1,100 DUC.budín c/frutas 170 g (z),100 DUCADOS,NaN,NaN,NaN,NaN,NaN,0
2,1,100 DUC.budín manz/can. 170 g (z),100 DUCADOS,NaN,NaN,NaN,NaN,NaN,0
3,1,100 DUC.budín maracuya 170 g (z),100 DUCADOS,NaN,NaN,NaN,NaN,NaN,0
4,1,100 DUC.pan dulce trad. 700 g (z),100 DUCADOS,NaN,NaN,NaN,NaN,NaN,0
5,1,100 DUC.torta cioccolato 600 g (z),100 DUCADOS,NaN,NaN,NaN,NaN,NaN,0
6,1,100 DUCADOS budin c/chips choc. 300 g,100 DUCADOS,NaN,NaN,NaN,NaN,NaN,0
7,1,100 DUCADOS budin c/chips ddl 300 g,100 DUCADOS,NaN,NaN,NaN,NaN,NaN,0
8,1,100 DUCADOS budin c/dulce de leche 300 g,100 DUCADOS,NaN,NaN,NaN,NaN,NaN,0
9,1,100 DUCADOS budin c/frut.secos 300 g,100 DUCADOS,NaN,NaN,NaN,NaN,NaN,0



Exportado: /content/productos_precios_promedio.xlsx


In [8]:
# Palabras clave a buscar (case-insensitive)
PALABRAS_CLAVE = ["lechuga", "ensalada", "hojas"]

# Columnas a traer de cada DataFrame
COLS_PRODUCTO = [
    "id_comercio", "id_bandera", "id_sucursal",
    "productos_ean",
    "productos_descripcion",
    "productos_marca",
    "productos_cantidad_presentacion",
    "productos_unidad_medida_presentacion",
    "productos_precio_lista",
    "productos_precio_referencia",
    "productos_precio_unitario_promo1",
    "productos_leyenda_promo1",
    "productos_precio_unitario_promo2",
    "productos_leyenda_promo2",
    "dia",
    "fecha",
]

COLS_COMERCIO = [
    "id_comercio", "id_bandera",
    "comercio_razon_social",
    "comercio_bandera_nombre",
]

COLS_SUCURSAL = [
    "id_comercio", "id_bandera", "id_sucursal",
    "sucursales_nombre",
    "sucursales_tipo",
    "sucursales_calle",
    "sucursales_numero",
    "sucursales_localidad",
    "sucursales_provincia",
    "sucursales_barrio",
    "sucursales_codigo_postal",
    "sucursales_latitud",
    "sucursales_longitud",
]

# Filtrar solo columnas que existen en cada DataFrame
cols_prod = [c for c in COLS_PRODUCTO if c in df_productos.columns]
cols_com  = [c for c in COLS_COMERCIO if c in df_comercios.columns]
cols_suc  = [c for c in COLS_SUCURSAL if c in df_sucursales.columns]

# ── 1. Normalizar claves de join a string en los tres DataFrames ──
CLAVES_JOIN = ["id_comercio", "id_bandera", "id_sucursal"]

for col in CLAVES_JOIN:
    for df in [df_productos, df_comercios, df_sucursales]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()

# ── 2. Filtrar productos por palabras clave ───────────────────
patron = "|".join(PALABRAS_CLAVE)

mask = df_productos["productos_descripcion"].str.contains(
    patron, case=False, na=False
)

df_filtrado = df_productos.loc[mask, cols_prod].copy()
print(f"Registros encontrados: {len(df_filtrado):,}")

if df_filtrado.empty:
    print("No se encontraron productos con las palabras clave indicadas.")
else:
    # ── 3. Join con comercios ─────────────────────────────────
    df_result = df_filtrado.merge(
        df_comercios[cols_com].drop_duplicates(),
        on=["id_comercio", "id_bandera"],
        how="left",
    )

    # ── 4. Join con sucursales ────────────────────────────────
    df_result = df_result.merge(
        df_sucursales[cols_suc].drop_duplicates(),
        on=["id_comercio", "id_bandera", "id_sucursal"],
        how="left",
    )

    # ── 5. Ordenar columnas en un orden legible ───────────────
    orden_columnas = [
        # Identificación geográfica
        "sucursales_provincia",
        "sucursales_localidad",
        "sucursales_barrio",
        "sucursales_codigo_postal",
        "sucursales_latitud",
        "sucursales_longitud",
        # Comercio
        "comercio_razon_social",
        "comercio_bandera_nombre",
        "sucursales_nombre",
        "sucursales_tipo",
        "sucursales_calle",
        "sucursales_numero",
        # Producto
        "productos_ean",
        "productos_descripcion",
        "productos_marca",
        "productos_cantidad_presentacion",
        "productos_unidad_medida_presentacion",
        # Precios
        "productos_precio_lista",
        "productos_precio_referencia",
        "productos_precio_unitario_promo1",
        "productos_leyenda_promo1",
        "productos_precio_unitario_promo2",
        "productos_leyenda_promo2",
        # Trazabilidad
        "dia",
        "fecha",
        # IDs (al final, como referencia)
        "id_comercio",
        "id_bandera",
        "id_sucursal",
    ]

    cols_finales = [c for c in orden_columnas if c in df_result.columns]
    df_result = (
        df_result[cols_finales]
        .sort_values(["sucursales_provincia", "sucursales_localidad",
                      "comercio_bandera_nombre", "productos_descripcion"])
        .reset_index(drop=True)
    )

    print(f"Filas en resultado final: {len(df_result):,}")
    print(f"Provincias cubiertas:     {df_result['sucursales_provincia'].nunique()}")
    print(f"Sucursales únicas:        {df_result['id_sucursal'].nunique()}")
    print(f"Productos únicos (EAN):   {df_result['productos_ean'].nunique()}")
    display(df_result.head(20))

    # ── 6. Exportar a Excel ───────────────────────────────────
    output_path = BASE_DIR / "verduras_hoja_por_sucursal.xlsx"

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        df_result.to_excel(writer, index=False, sheet_name="Productos")

        ws = writer.sheets["Productos"]

        # Ancho automático de columnas
        for col in ws.columns:
            max_len = max(len(str(cell.value or "")) for cell in col)
            ws.column_dimensions[col[0].column_letter].width = min(max_len + 4, 50)

        # Formato numérico para precios y coordenadas
        col_indices = {cell.value: cell.column for cell in ws[1]}

        cols_precio = [
            "productos_precio_lista",
            "productos_precio_referencia",
            "productos_precio_unitario_promo1",
            "productos_precio_unitario_promo2",
        ]
        cols_coord = ["sucursales_latitud", "sucursales_longitud"]

        for nombre_col in cols_precio:
            if nombre_col in col_indices:
                for row in ws.iter_rows(min_row=2,
                                        min_col=col_indices[nombre_col],
                                        max_col=col_indices[nombre_col]):
                    for cell in row:
                        cell.number_format = "#,##0.00"

        for nombre_col in cols_coord:
            if nombre_col in col_indices:
                for row in ws.iter_rows(min_row=2,
                                        min_col=col_indices[nombre_col],
                                        max_col=col_indices[nombre_col]):
                    for cell in row:
                        cell.number_format = "0.000000"

    print(f"\nExportado: {output_path}")

Registros encontrados: 196
Filas en resultado final: 196
Provincias cubiertas:     21
Sucursales únicas:        53
Productos únicos (EAN):   1


,sucursales_provincia,sucursales_localidad,sucursales_barrio,sucursales_codigo_postal,sucursales_latitud,sucursales_longitud,comercio_razon_social,comercio_bandera_nombre,sucursales_nombre,sucursales_tipo,...,productos_precio_referencia,productos_precio_unitario_promo1,productos_leyenda_promo1,productos_precio_unitario_promo2,productos_leyenda_promo2,dia,fecha,id_comercio,id_bandera,id_sucursal
0,AR-A,SALTA,...,,-24.818818,-65.425411,YAGUAR SA,MAYORISTA YAGUAR,Salta,Mayorista,...,NaN,NaN,NaN,NaN,NaN,martes,2026-04-21,64,1,9
1,AR-A,Salta,NaN,4414,-24.819504,-65.425206,Autoservicio Mayorista Diarco S.A.,Diarco,Salta,Mayorista,...,NaN,NaN,NaN,NaN,NaN,martes,2026-04-21,65,1,26
2,AR-A,Salta,NaN,4414,-24.819504,-65.425206,Autoservicio Mayorista Diarco S.A.,Diarco,Salta,Mayorista,...,NaN,NaN,NaN,NaN,NaN,martes,2026-04-21,65,1,26
3,AR-A,Salta,NaN,4414,-24.819504,-65.425206,Autoservicio Mayorista Diarco S.A.,Diarco,Salta,Mayorista,...,NaN,NaN,NaN,NaN,NaN,martes,2026-04-21,65,1,26
4,AR-B,9 de Julio,NaN,6500,-35.470083,-60.851771,Autoservicio Mayorista Diarco S.A.,Diarco,9 de Julio,Mayorista,...,NaN,NaN,NaN,NaN,NaN,martes,2026-04-21,65,1,22
5,AR-B,9 de Julio,NaN,6500,-35.470083,-60.851771,Autoservicio Mayorista Diarco S.A.,Diarco,9 de Julio,Mayorista,...,NaN,NaN,NaN,NaN,NaN,martes,2026-04-21,65,1,22
6,AR-B,9 de Julio,NaN,6500,-35.470083,-60.851771,Autoservicio Mayorista Diarco S.A.,Diarco,9 de Julio,Mayorista,...,NaN,NaN,NaN,NaN,NaN,martes,2026-04-21,65,1,22
7,AR-B,B.Blanca,NaN,8000,-38.739688,-62.295420,Autoservicio Mayorista Diarco S.A.,Diarco,Bahia Blanca,Mayorista,...,NaN,NaN,NaN,NaN,NaN,martes,2026-04-21,65,1,11
8,AR-B,B.Blanca,NaN,8000,-38.739688,-62.295420,Autoservicio Mayorista Diarco S.A.,Diarco,Bahia Blanca,Mayorista,...,NaN,NaN,NaN,NaN,NaN,martes,2026-04-21,65,1,11
9,AR-B,B.Blanca,NaN,8000,-38.739688,-62.295420,Autoservicio Mayorista Diarco S.A.,Diarco,Bahia Blanca,Mayorista,...,NaN,NaN,NaN,NaN,NaN,martes,2026-04-21,65,1,11



Exportado: /content/verduras_hoja_por_sucursal.xlsx
